# 06 — Análise de Sentimento dos Reviews

Este notebook calcula sentiment score com VADER usando `review_title` + `review_content`, cria labels (positivo/neutro/negativo), compara com rating numérico e gera WordClouds.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Carregar base processada (review-level)

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'base_processada.csv')
df.shape


## Sentiment score + labels

In [ ]:
analyzer = SentimentIntensityAnalyzer()
text = (df['review_title'].fillna('') + ' ' + df['review_content'].fillna('')).str.strip()

df_out = df.copy()
df_out['sentiment_score'] = text.map(lambda t: analyzer.polarity_scores(t)['compound'])

def sentiment_label(x: float) -> str:
    if x <= -0.05:
        return 'negativo'
    if x >= 0.05:
        return 'positivo'
    return 'neutro'

df_out['sentimento_label'] = df_out['sentiment_score'].map(sentiment_label)
df_out[['review_title','sentiment_score','sentimento_label']].head(5)


## Divergência sentimento vs rating

In [ ]:
df_out['rating_clean'] = pd.to_numeric(df_out.get('rating_clean', df_out.get('rating')), errors='coerce')

def rating_bucket(r: float) -> str | None:
    if pd.isna(r):
        return None
    if r >= 4.0:
        return 'alto'
    if r <= 2.5:
        return 'baixo'
    return 'medio'

df_out['rating_bucket'] = df_out['rating_clean'].map(rating_bucket)
ct = pd.crosstab(df_out['sentimento_label'], df_out['rating_bucket'], normalize='columns')
ct


## WordCloud por sentimento

In [ ]:
def make_wordcloud(text_series: pd.Series, path: Path, *, title: str) -> None:
    corpus = ' '.join(text_series.dropna().astype(str).tolist())
    wc = WordCloud(width=1400, height=700, background_color='black', collocations=False).generate(corpus)
    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

positive = (df_out.loc[df_out['sentimento_label'] == 'positivo', 'review_title'].fillna('') + ' ' + df_out.loc[df_out['sentimento_label'] == 'positivo', 'review_content'].fillna('')).str.strip()
negative = (df_out.loc[df_out['sentimento_label'] == 'negativo', 'review_title'].fillna('') + ' ' + df_out.loc[df_out['sentimento_label'] == 'negativo', 'review_content'].fillna('')).str.strip()

make_wordcloud(positive, FIGURES_DIR / 'wordcloud_positive.png', title='WordCloud: Reviews Positivos')
make_wordcloud(negative, FIGURES_DIR / 'wordcloud_negative.png', title='WordCloud: Reviews Negativos')


## Exportar base com sentimento

In [ ]:
df_out.to_csv(PROCESSED_DIR / 'base_com_sentimento.csv', index=False)
df_out[['main_category','sentimento_label']].value_counts().reset_index(name='n').head(10)
